In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Load and Preprocess Data
df = pd.read_csv('smart_grid_data.csv')
df = df.fillna(method='ffill') # Fill missing values

# Assume target column is 'status' (0 for normal, 1 for anomaly/fault)
# Or if it's purely unsupervised, we filter by a normal household/condition
normal_df = df[df['status'] == 0].drop('status', axis=1)
anomaly_df = df[df['status'] == 1].drop('status', axis=1)

# Scale features
scaler = StandardScaler()
normal_scaled = scaler.fit_transform(normal_df)
anomaly_scaled = scaler.transform(anomaly_df)

# Helper function to create time-series windows: shape becomes (samples, timesteps, features)
def create_windows(data, timesteps=10):
    windows = []
    for i in range(len(data) - timesteps):
        windows.append(data[i:(i + timesteps)])
    return np.array(windows)

TIMESTEPS = 10
normal_windows = create_windows(normal_scaled, timesteps=TIMESTEPS)
anomaly_windows = create_windows(anomaly_scaled, timesteps=TIMESTEPS)

# Split normal windows into train and validation sets
X_train, X_test_normal = train_test_split(normal_windows, test_size=0.2, random_state=42)

n_features = X_train.shape[2]

# 2. Build LSTM Autoencoder Architecture
model = keras.Sequential([
    # Encoder
    layers.Input(shape=(TIMESTEPS, n_features)),
    layers.LSTM(32, activation='relu', return_sequences=False),
    
    # Bridge to repeat vector across timesteps
    layers.RepeatVector(TIMESTEPS),
    
    # Decoder
    layers.LSTM(32, activation='relu', return_sequences=True),
    layers.TimeDistributed(layers.Dense(n_features))
])

model.compile(optimizer='adam', loss='mse')

# 3. Train strictly on normal sequential data
H = model.fit(X_train, X_train, epochs=10, batch_size=32, validation_data=(X_test_normal, X_test_normal))

# 4. Set Threshold using normal validation windows
X_test_preds = model.predict(X_test_normal)
# Mean squared error averaged across timesteps and features
normal_mse = np.mean(np.power(X_test_normal - X_test_preds, 2), axis=(1, 2))
threshold = np.percentile(normal_mse, 99)

# 5. Evaluate on Anomaly Windows
anomaly_preds = model.predict(anomaly_windows)
anomaly_mse = np.mean(np.power(anomaly_windows - anomaly_preds, 2), axis=(1, 2))

anomalies = anomaly_mse > threshold
print(f"Total Detected Smart Grid Anomalies: {sum(anomalies)}")